In [ ]:
# 1
# Necessary install packages + Personal NGROK authentication ID
!pip install flask transformers torch pyngrok sentence-transformers pandas sacremoses trl
!ngrok config add-authtoken 2tiD7xznMIuyWhuOvgetvnnpGBN_8agtxVyhtshgkrBDUPdR6

In [21]:
# 2
# Packages + Mounting Drive
from flask import Flask, request, render_template_string, jsonify
import torch
from transformers import BioGptTokenizer, BioGptForCausalLM
from transformers import pipeline, set_seed, AutoTokenizer, AutoModelForSequenceClassification
from pyngrok import ngrok
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import sqlite3
from datetime import datetime
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
drive_path = '/content/drive/My Drive/MedBot_Data'
os.makedirs(drive_path, exist_ok=True)

if not os.path.exists(f'{drive_path}/data.csv'):
    raise FileNotFoundError(f"Please upload data.csv to {drive_path} via Google Drive website")

Mounted at /content/drive


In [22]:
# 3
# Model Initialization
embedding_model = SentenceTransformer('cambridgeltl/SapBERT-from-PubMedBERT-fulltext')
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
tokenizer = BioGptTokenizer.from_pretrained("microsoft/biogpt")
model = BioGptForCausalLM.from_pretrained("microsoft/biogpt")

Device set to use cpu


In [23]:
# 4
# FAQ Database Embedding
faq_path = f'{drive_path}/data.csv'
faq_df = pd.read_csv(faq_path)
questions = faq_df['Question'].tolist()
answers = faq_df['Answer'].tolist()

question_embeddings = embedding_model.encode(questions, convert_to_tensor=True)

In [5]:
# 5
# Is query medical function
def is_medical(query):
    candidate_labels = ["medical", "non-medical"]
    result = classifier(query, candidate_labels, multi_label=False)
    return result['labels'][0] == 'medical' and result['scores'][0] > 0.65

In [6]:
# 6
# Generate response function
def generate_response(user_input, threshold=0.75):
    input_embedding = embedding_model.encode(user_input, convert_to_tensor=True)
    cos_scores = util.pytorch_cos_sim(input_embedding, question_embeddings)[0]

    top_score = torch.max(cos_scores).item()
    top_idx = torch.argmax(cos_scores)
    context = answers[top_idx]

    if top_score > threshold:
      return context

    if not is_medical(user_input):
        return "I specialize in prostate cancer information. Please ask medical questions related to this field."


    prompt = f"""As a clinical AI assistant specializing in oncology, respond to this medical query:

    Query: {user_input}

    Guidelines:
    1. Provide accurate, evidence-based information
    2. Acknowledge when queries are outside prostate cancer focus
    3. Recommend consulting a physician for personal medical advice

    Response:"""


    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=1024,
        truncation=True,
        add_special_tokens=True
    )

    set_seed(42)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.2,
            num_return_sequences=1,
            pad_token_id=tokenizer.eos_token_id
        )

    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_response.split("Response:")[-1].strip()

In [27]:
# 7
# Initialization of application + HTML Template

app = Flask(__name__)

HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>MedBot Assistant</title>
    <style>
        :root {
            --primary: #4A90E2;
            --secondary: #6C5CE7;
            --background: #F8F9FA;
        }

        body {
            max-width: 800px;
            margin: 20px auto;
            padding: 40px 20px;
            font-family: 'Segoe UI', system-ui, sans-serif;
            background: var(--background);
        }

        .header {
            text-align: center;
            margin-bottom: 40px;
            position: relative;
        }

        .robot {
            width: 120px;
            margin: -80px auto 20px;
            animation: float 3s ease-in-out infinite;
        }

        h1 {
            color: var(--secondary);
            font-size: 2.5rem;
            margin: 0 0 10px;
            background: linear-gradient(45deg, var(--primary), var(--secondary));
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
        }

        form {
            background: white;
            padding: 30px;
            border-radius: 20px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.08);
            margin-bottom: 30px;
        }

        input[type="text"] {
            width: 100%;
            padding: 15px;
            border: 2px solid #eee;
            border-radius: 12px;
            font-size: 1rem;
            margin-bottom: 15px;
            transition: border-color 0.3s ease;
        }

        input[type="text"]:focus {
            outline: none;
            border-color: var(--primary);
        }

        input[type="submit"] {
            background: linear-gradient(45deg, var(--primary), var(--secondary));
            color: white;
            padding: 15px 30px;
            border: none;
            border-radius: 12px;
            font-weight: bold;
            cursor: pointer;
            transition: transform 0.2s ease;
        }

        input[type="submit"]:hover {
            transform: translateY(-2px);
        }

        .response {
            background: white;
            padding: 25px;
            border-radius: 15px;
            box-shadow: 0 5px 15px rgba(0,0,0,0.05);
            margin-top: 20px;
            position: relative;
            animation: fadeIn 0.5s ease-out;
        }

        .response:before {
            content: '';
            position: absolute;
            top: -10px;
            left: 30px;
            border-left: 10px solid transparent;
            border-right: 10px solid transparent;
            border-bottom: 10px solid white;
        }

        .disclaimer {
            color: #666;
            font-size: 0.85rem;
            margin-top: 25px;
            padding-top: 15px;
            border-top: 1px solid #eee;
        }

        @keyframes float {
            0%, 100% { transform: translateY(0); }
            50% { transform: translateY(-10px); }
        }

        @keyframes fadeIn {
            from { opacity: 0; transform: translateY(10px); }
            to { opacity: 1; transform: translateY(0); }
        }
    </style>
</head>
<body>
    <div class="header">
        <img src="https://cdn-icons-png.flaticon.com/512/1041/1041916.png" class="robot" alt="MedBot Assistant">
        <h1>BioGPT Prostate Cancer Medical Chatbot</h1>
        <p>Your AI-powered prostate cancer information resource</p>
    </div>

    <form method="POST">
        <input type="text" name="user_input" value="{{ user_input }}" placeholder="Ask me about prostate cancer..." required>
        <input type="submit" value="Get Answer">
    </form>

    {% if response %}
    <div class="response">
        <h3>Here's what I found:</h3>
        <p>{{ response }}</p>
        <div class="disclaimer">
            ℹ️ Note: This AI assistant provides general information only. Always consult a healthcare professional for medical advice.
        </div>
    </div>
    <div class="feedback">
    <span>Was this helpful?</span>
    <button onclick="submitFeedback('thumbs_up')">👍</button>
    <button onclick="submitFeedback('thumbs_down')">👎</button>
</div>


<script>
    function submitFeedback(type) {
        const userInput = document.querySelector('input[name="user_input"]').value;
        const responseText = document.querySelector('.response p').innerText;

        fetch('/feedback', {
            method: 'POST',
            headers: {'Content-Type': 'application/json'},
            body: JSON.stringify({
                input: userInput,
                response: responseText,
                feedback: type
            })
        });

        document.querySelector('.feedback').innerHTML = 'Thank you for your feedback!';
    }
    </script>




    {% endif %}
</body>
</html>
"""

In [14]:
# 8
# Create feedback.db database if not already created
def init_feedback_db():
    db_path = f'{drive_path}/feedback.db'
    conn = sqlite3.connect(db_path)
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS feedback
                 (id INTEGER PRIMARY KEY AUTOINCREMENT,
                  input_text TEXT,
                  response_text TEXT,
                  feedback TEXT,
                  timestamp DATETIME)''')
    conn.commit()
    conn.close()

init_feedback_db()

In [ ]:
# 9
# View last 10 queries and their feedback
def view_feedback():
    db_path = f'/content/drive/My Drive/MedBot_Data/feedback.db'
    conn = sqlite3.connect(db_path)
    df = pd.read_sql("SELECT * FROM feedback", conn)
    conn.close()
    return df

view_feedback().tail(10)

In [28]:
# 10
# Begin hosting NGROK website

@app.route("/", methods=["GET", "POST"])
def home():
    if request.method == "POST":
        user_input = request.form["user_input"]
        response = generate_response(user_input)
        return render_template_string(HTML_TEMPLATE, response=response, user_input=user_input)
    return render_template_string(HTML_TEMPLATE)


@app.route("/feedback", methods=["POST"])
def handle_feedback():
    data = request.json
    try:
        db_path = f'{drive_path}/feedback.db'
        conn = sqlite3.connect(db_path)
        c = conn.cursor()
        c.execute('''INSERT INTO feedback
                    (input_text, response_text, feedback, timestamp)
                    VALUES (?, ?, ?, ?)''',
                    (data['input'], data['response'],
                     data['feedback'], datetime.now()))
        conn.commit()
        return jsonify(success=True)
    except Exception as e:
        print(f"Error saving feedback: {e}")
        return jsonify(success=False, error=str(e)), 500
    finally:
        conn.close()


ngrok_tunnel = ngrok.connect(5000)
print("Public URL:", ngrok_tunnel.public_url)
app.run(host='0.0.0.0', debug=False)

Public URL: https://d868-35-185-70-144.ngrok-free.app
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
